# END_DIP — Traffic Safety Monitoring (DIP + YOLO)
Run top-to-bottom in Google Colab. Enable a GPU first.

This notebook stores full-resolution results on Google Drive and creates a smaller browser-compatible preview in `/content` for reliable inline playback.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!rm -rf /content/DIP
!git clone -b feature/yolo-traffic-safety https://github.com/NVTruong473/DIP.git /content/DIP
%cd /content/DIP/END_DIP


In [ ]:
!pip install -q -r requirements.txt


In [ ]:
import os, torch
VIDEO='/content/drive/MyDrive/DIP/video1.mp4'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Video exists:', os.path.exists(VIDEO), VIDEO)
assert os.path.exists(VIDEO), 'Put video1.mp4 in MyDrive/DIP first.'


In [ ]:
!python download_models.py --models-dir '/content/drive/MyDrive/DIP/models'


## Process the default Drive video
The full-quality output stays persistent under `MyDrive/DIP/outputs/`.

In [ ]:
!python main.py \
  --input '/content/drive/MyDrive/DIP/video1.mp4' \
  --output-dir '/content/drive/MyDrive/DIP/outputs' \
  --models-dir '/content/drive/MyDrive/DIP/models'


## Preview result directly in Colab
Do **not** point the HTML video player directly at a mounted Drive path. This cell makes a lightweight H.264/yuv420p preview in `/content` and embeds it into the notebook. The original full-resolution result remains on Drive.

In [ ]:
import os
import subprocess
from IPython.display import Video, display

DRIVE_OUT = '/content/drive/MyDrive/DIP/outputs/video1_result.mp4'
PREVIEW = '/content/video1_result_preview.mp4'

assert os.path.exists(DRIVE_OUT), f'Output not found: {DRIVE_OUT}'

cmd = [
    'ffmpeg', '-y', '-loglevel', 'error',
    '-i', DRIVE_OUT,
    '-vf', 'scale=960:-2',
    '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '27',
    '-pix_fmt', 'yuv420p', '-tag:v', 'avc1',
    '-movflags', '+faststart', '-an', PREVIEW,
]
subprocess.run(cmd, check=True)

print(f'Full result : {DRIVE_OUT} ({os.path.getsize(DRIVE_OUT)/1024/1024:.1f} MB)')
print(f'Colab preview: {PREVIEW} ({os.path.getsize(PREVIEW)/1024/1024:.1f} MB)')
display(Video(PREVIEW, embed=True, width=900, html_attributes='controls'))


### Re-open the preview later without re-processing the YOLO pipeline
If the preview cell was already run in the current runtime, use the next cell.

In [ ]:
from IPython.display import Video, display
display(Video('/content/video1_result_preview.mp4', embed=True, width=900, html_attributes='controls'))


## Optional: Gradio UI for another video
Run this cell, then open the public Gradio link and upload any replacement video. The output encoder is also forced to H.264/yuv420p for browser playback.

In [ ]:
!python app.py


## Optional fine-tuning
Only run this if you want a custom `helmet_best.pt`; normal inference already works with the public pretrained checkpoint.

In [ ]:
# import os
# os.environ['ROBOFLOW_API_KEY'] = 'YOUR_KEY'
# !python training/prepare_roboflow_dataset.py --target '/content/drive/MyDrive/DIP/datasets/helmet_rf_v5'
# !python training/train_helmet.py --data '/content/drive/MyDrive/DIP/datasets/helmet_rf_v5/data_helmet_2class.yaml' --models-dir '/content/drive/MyDrive/DIP/models' --runs-dir '/content/drive/MyDrive/DIP/training_runs' --epochs 20 --batch 16
